In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. 把数据和代码复制到working目录下，进入目录，安装依赖

In [1]:
# repo从input（只读）复制到working（可写）
!cp -r /kaggle/input/datasets/huchenkai12/dataset247/emg2qwerty-main /kaggle/working/emg2qwerty
# 进入working（可写）
%cd /kaggle/working/emg2qwerty
# 安装环境
!pip -q install -r requirements.txt
# 后面即可修改代码

/kaggle/working/emg2qwerty
     - 553.6 kB 10.2 MB/s 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 110.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.2/437.2 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.8/300.8 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.6/163.6 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 77.1 MB/s eta 0:00:00
 

# 2. 测试baseline训练，确认是否可以跑通

In [2]:
# baseline的训练1轮， 测试是否可以跑通
!python -m emg2qwerty.train \
  user=single_user \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  trainer.max_epochs=1 \
  dataset.root=/kaggle/input/datasets/huchenkai12/data247/89335547 \
  hydra.run.dir=/kaggle/working/logs/${now:%Y-%m-%d}/${now:%H-%M-%S}

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.0
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[2026-03-08 13:00:35,866][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-

# 3. CHENKAI CNN+RNN HYBRID VERSION： 追加新的 CNNRNNCTCModule

## 刷新目录，确保 lightning.py 纯净

In [6]:
!rm -rf emg2qwerty
!cp -r /kaggle/input/datasets/huchenkai12/dataset247/emg2qwerty-main/emg2qwerty .


from pathlib import Path

p = Path("emg2qwerty/lightning.py")
src = p.read_text()

if "class CNNRNNCTCModule" not in src:
    print("目录刷新，CNNRNNCTCModule 已清除。")

目录刷新，CNNRNNCTCModule 已清除。


## 追加 CNNRNNCTCModule

In [7]:
from pathlib import Path

p = Path("emg2qwerty/lightning.py")

append_code = """
# ===== 新增：CNN + RNN =====
class CNNRNNCTCModule(pl.LightningModule):

    NUM_BANDS: ClassVar[int] = 2
    ELECTRODE_CHANNELS: ClassVar[int] = 16

    def __init__(
        self,
        in_features: int,
        mlp_features: Sequence[int],
        cnn_channels: int,
        kernel_size: int,
        num_cnn_layers: int,
        rnn_hidden_size: int,
        rnn_layers: int,
        dropout: float,
        bidirectional: bool,
        optimizer: DictConfig,
        lr_scheduler: DictConfig,
        decoder: DictConfig,
    ) -> None:
        super().__init__()
        self.save_hyperparameters()

        # 前端和原 CNN 版本保持一致
        num_features = self.NUM_BANDS * mlp_features[-1]  # e.g. 2 * 384 = 768

        self.frontend = nn.Sequential(
            SpectrogramNorm(channels=self.NUM_BANDS * self.ELECTRODE_CHANNELS),
            MultiBandRotationInvariantMLP(
                in_features=in_features,
                mlp_features=mlp_features,
                num_bands=self.NUM_BANDS,
            ),
            nn.Flatten(start_dim=2),  # (T, N, F)
        )

        # CNN 提局部时序特征
        layers = []
        in_ch = num_features
        
        for i in range(num_cnn_layers):
            stride = 2 if i == 0 else 1
            layers.append(
                nn.Conv1d(
                    in_channels=in_ch,
                    out_channels=cnn_channels,
                    kernel_size=kernel_size,
                    stride=stride,
                    padding=kernel_size // 2,
                )
            )
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_ch = cnn_channels

        self.temporal_cnn = nn.Sequential(*layers)

        # RNN 建模长时依赖
        self.rnn = nn.GRU(
            input_size=cnn_channels,
            hidden_size=rnn_hidden_size,
            num_layers=rnn_layers,
            dropout=dropout if rnn_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )

        rnn_out_features = rnn_hidden_size * (2 if bidirectional else 1)

        # CTC head
        self.classifier = nn.Sequential(
            nn.Linear(rnn_out_features, charset().num_classes),
            nn.LogSoftmax(dim=-1),
        )

        self.ctc_loss = nn.CTCLoss(blank=charset().null_class)
        self.decoder = instantiate(decoder)

        metrics = MetricCollection([CharacterErrorRates()])
        self.metrics = nn.ModuleDict(
            {
                f"{phase}_metrics": metrics.clone(prefix=f"{phase}/")
                for phase in ["train", "val", "test"]
            }
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        # inputs: (T, N, bands=2, C=16, freq)
        x = self.frontend(inputs)      # (T, N, F)
        x = x.permute(1, 2, 0)         # (N, F, T)
        x = self.temporal_cnn(x)       # (N, C, T)
        x = x.permute(2, 0, 1)         # (T, N, C)
        x, _ = self.rnn(x)             # (T, N, H) or (T, N, 2H)
        emissions = self.classifier(x) # (T, N, num_classes)
        return emissions

    def _step(self, phase: str, batch: dict[str, torch.Tensor], *args, **kwargs) -> torch.Tensor:
        inputs = batch["inputs"]
        targets = batch["targets"]
        input_lengths = batch["input_lengths"]
        target_lengths = batch["target_lengths"]
        N = len(input_lengths)

        emissions = self.forward(inputs)

        # CNN padding 保持时间长度不变，GRU 也不改时间长度
        emission_lengths = torch.div(input_lengths + 1, 2, rounding_mode="floor")

        loss = self.ctc_loss(
            log_probs=emissions,
            targets=targets.transpose(0, 1),
            input_lengths=emission_lengths,
            target_lengths=target_lengths,
        )

        predictions = self.decoder.decode_batch(
            emissions=emissions.detach().cpu().numpy(),
            emission_lengths=emission_lengths.detach().cpu().numpy(),
        )

        metrics = self.metrics[f"{phase}_metrics"]
        targets_np = targets.detach().cpu().numpy()
        target_lengths_np = target_lengths.detach().cpu().numpy()
        for i in range(N):
            target = LabelData.from_labels(targets_np[: target_lengths_np[i], i])
            metrics.update(prediction=predictions[i], target=target)

        self.log(f"{phase}/loss", loss, batch_size=N, sync_dist=True)
        return loss

    def _epoch_end(self, phase: str) -> None:
        metrics = self.metrics[f"{phase}_metrics"]
        self.log_dict(metrics.compute(), sync_dist=True)
        metrics.reset()

    def training_step(self, *args, **kwargs) -> torch.Tensor:
        return self._step("train", *args, **kwargs)

    def validation_step(self, *args, **kwargs) -> torch.Tensor:
        return self._step("val", *args, **kwargs)

    def test_step(self, *args, **kwargs) -> torch.Tensor:
        return self._step("test", *args, **kwargs)

    def on_train_epoch_end(self) -> None:
        self._epoch_end("train")

    def on_validation_epoch_end(self) -> None:
        self._epoch_end("val")

    def on_test_epoch_end(self) -> None:
        self._epoch_end("test")

    def configure_optimizers(self) -> dict[str, Any]:
        return utils.instantiate_optimizer_and_scheduler(
            self.parameters(),
            optimizer_config=self.hparams.optimizer,
            lr_scheduler_config=self.hparams.lr_scheduler,
        )
"""

p.write_text(p.read_text() + "\\n" + append_code)
print("✅ 已追加 CNNRNNCTCModule 到 emg2qwerty/lightning.py")

✅ 已追加 CNNRNNCTCModule 到 emg2qwerty/lightning.py


## 修复 lightning.py 文件格式（字面量 \n）

In [8]:
from pathlib import Path

p = Path("emg2qwerty/lightning.py")
txt = p.read_text()

txt = txt.replace("\\n", "\n")

p.write_text(txt)
print("✅ 已修复字面量 \\n")

✅ 已修复字面量 \n


# 4.设置模型参数，以YAML文件形式保存

In [9]:
from pathlib import Path

p = Path("config/model/cnn_rnn_ctc.yaml")
p.write_text(
"""# @package _global_
module:
  _target_: emg2qwerty.lightning.CNNRNNCTCModule
  in_features: 528
  mlp_features: [384]

  cnn_channels: 256
  kernel_size: 7
  num_cnn_layers: 2

  rnn_hidden_size: 128
  rnn_layers: 1
  bidirectional: true

  dropout: 0.1

datamodule:
  _target_: emg2qwerty.lightning.WindowedEMGDataModule
  window_length: 8000
  padding: [1800, 200]
"""
)
print("✅ 已创建 config/model/cnn_rnn_ctc.yaml")

✅ 已创建 config/model/cnn_rnn_ctc.yaml


## 检查 YAML 文件是否成功创建

In [10]:
from pathlib import Path

p = Path("config/model/cnn_rnn_ctc.yaml")
print(p.exists())

True


# 5. CNN+RNN方法: SMOKE TESTs（调整了LR，现训练10epochs）

In [65]:
!python -m emg2qwerty.train \
  user=single_user \
  model=cnn_rnn_ctc \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  trainer.max_epochs=10 \
  optimizer.lr=0.0005 \
  dataset.root=/kaggle/input/datasets/huchenkai12/data247/89335547 \
  hydra.run.dir=/kaggle/working/logs/${now:%Y-%m-%d}/${now:%H-%M-%S}

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.0
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[2026-03-08 11:51:21,549][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-

# 6. FORMAL TEST - 40 EPOCHS (NEW)

In [11]:
!python -m emg2qwerty.train \
  user=single_user \
  model=cnn_rnn_ctc \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  trainer.max_epochs=40 \
  optimizer.lr=0.0005 \
  dataset.root=/kaggle/input/datasets/huchenkai12/data247/89335547 \
  hydra.run.dir=/kaggle/working/logs/${now:%Y-%m-%d}/${now:%H-%M-%S}

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.0
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[2026-03-08 13:06:53,297][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-

# 附录. (OLD VERSION) FORMAL TEST - 40 EPOCHS 

In [27]:
!python -m emg2qwerty.train \
  user=single_user \
  model=cnn_rnn_ctc \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  trainer.max_epochs=40 \
  dataset.root=/kaggle/input/datasets/huchenkai12/data247/89335547 \
  hydra.run.dir=/kaggle/working/logs/${now:%Y-%m-%d}/${now:%H-%M-%S}

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.0
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[2026-03-08 10:21:37,500][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-